# 06 - IEEE-CIS Logical Rule Ablation

In [ ]:
from pathlib import Path
import json
import os
import sys

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
KAGGLE = Path("/kaggle").exists()
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)
print({"project_root": str(PROJECT_ROOT), "quick_run": QUICK_RUN, "kaggle": KAGGLE})

In [ ]:
from src.experiment import run_predictive_benchmarks
from src.logic import FraudRuleEngine

output_dir = OUTPUT_BASE / "05_rule_ablation"
result = run_predictive_benchmarks(
    PROJECT_ROOT / "configs/ieee_cis.yaml",
    output_dir=output_dir,
    model_names=("tree",),
    quick_run=QUICK_RUN,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
)
config = result["config"]
prepared = result["prepared"]
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
truth = engine.evaluate(prepared.test_frame)
probabilities = result["test_probabilities"]["tree"]
threshold = result["thresholds"]["tree"]
predicted_alert = probabilities >= threshold
activation = float(config["logic"]["activation_threshold"])
print({"data_source": result["data_source"], "rules": truth.columns.tolist()})

## Data

In [ ]:
display(truth.describe().T[["mean", "std", "min", "max"]].round(4))

## Results

In [ ]:
def score_rule_subset(name, columns):
    if columns:
        rule_score = truth[columns].max(axis=1).to_numpy(float)
    else:
        rule_score = np.zeros(len(truth), dtype=float)
    explained = rule_score >= activation
    explained_alert = explained & predicted_alert
    precision = prepared.y_test[explained_alert].mean() if explained_alert.any() else 0.0
    base_precision = prepared.y_test[predicted_alert].mean() if predicted_alert.any() else 0.0
    return {
        "ablation": name,
        "rule_count": len(columns),
        "coverage_all": explained.mean(),
        "coverage_alerts": explained_alert.sum() / max(predicted_alert.sum(), 1),
        "explained_alert_precision": precision,
        "precision_gain": precision - base_precision,
        "prediction_rule_consistency": (predicted_alert == explained).mean(),
    }

all_rules = truth.columns.tolist()
rows = [score_rule_subset("full_rule_set", all_rules), score_rule_subset("no_rules", [])]
rows.extend(score_rule_subset(f"only:{rule}", [rule]) for rule in all_rules)
rows.extend(score_rule_subset(f"without:{rule}", [item for item in all_rules if item != rule]) for rule in all_rules)
ablation = pd.DataFrame(rows).sort_values(["coverage_alerts", "precision_gain"], ascending=False)
display(ablation.round(4))

ablation.to_csv(output_dir / "rule_ablation.csv", index=False)

In [ ]:
plot_data = ablation[ablation["ablation"].str.startswith("without:")].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=plot_data, x="ablation", y="coverage_alerts", ax=axes[0], color="#4C72B0")
axes[0].tick_params(axis="x", rotation=35)
axes[0].set_title("Alert coverage after removing one rule")
sns.barplot(data=plot_data, x="ablation", y="precision_gain", ax=axes[1], color="#55A868")
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].tick_params(axis="x", rotation=35)
axes[1].set_title("Precision gain after removing one rule")
plt.tight_layout()
fig.savefig(output_dir / "rule_ablation.png", dpi=160, bbox_inches="tight")
plt.show()

## Takeaways

In [ ]:
full = ablation.query("ablation == 'full_rule_set'").iloc[0]
best = ablation[ablation["ablation"].str.startswith("without:")].sort_values("precision_gain", ascending=False).iloc[0]
display(Markdown(
    f"- Full-set alert coverage: **{full['coverage_alerts']:.3f}**.\n"
    f"- Full-set precision gain: **{full['precision_gain']:.3f}**.\n"
    f"- Highest leave-one-out precision gain: **{best['ablation']} = {best['precision_gain']:.3f}**.\n"
    "- Interpret ablation only after confirming real data and locked predictor probabilities."
))